In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

#importing the pre-trained model
model=models.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 136MB/s]


In [ ]:
SEED=42
import numpy as np
np.random.seed(SEED)
torch.manual_seed(SEED)

Freeze Convolution Layer

In [ ]:
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader

In [ ]:
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])
train_data=datasets.CIFAR10(root="./data",train=True,download=True,transform=transform)
test_data=datasets.CIFAR10(root="./data",train=False,download=True,transform=transform)

train_loader=DataLoader(train_data,batch_size=64,shuffle=True)
test_loader=DataLoader(test_data,batch_size=64)

100%|██████████| 170M/170M [00:03<00:00, 49.2MB/s]


In [ ]:
for param in model.features.parameters():
  param.require_grad=False

In [ ]:
#change the classifer
model.classifier[6]=nn.Linear(4096,10)

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [ ]:
import torch.optim as optim
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.classifier.parameters(),lr=0.001)

#Evaluation Loop

In [ ]:
def evaluate(model,loader,device):
  model.eval()
  correct=0
  total=0
  with torch.no_grad():
    for images,labels in loader:
      images,labels=images.to(device),labels.to(device)
      output=model(images)
      _,predicted=torch.max(output,1)
      total+=labels.size(0)
      correct+=(predicted==labels).sum().item()
    accuracy=100*correct/total
    return accuracy

# Traning Loop

In [ ]:
for epoch in range(1):
  model.train()
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
  print(f"Epoch: {epoch+1} loss: {loss.item()}")

Epoch: 1 loss: 1.9328806400299072


In [ ]:
acc1 = evaluate(model, test_loader, device)
print("Accuracy after feature extraction:", acc1)

Accuracy after feature extraction: 86.1


Unfreeze last convolution

In [ ]:
for param in model.features[-4:].parameters():
  param.requires_grad=True

In [ ]:
optimizer=optim.Adam(model.parameters(),lr=1e-5)

In [ ]:
for epoch in range(1):
  model.train()
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
  print(f"Epoch: {epoch+1} loss: {loss.item()}")

In [ ]:
acc2 = evaluate(model, test_loader, device)
print("Accuracy after feature extraction:", acc2)